<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/11_shap_explanability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11: SHAP explanations of the LightGBM models

**Purpose:** Explain the per-crop LightGBM models with SHAP, grouped by feature family (climate, soil, terrain, salinity, and sentinel and modis if they matter), and test whether the Sentinel and MODIS features add signal under the same spatial folds. The explanations feed the `recommend()` function (notebook 12).

**Inputs:** `training_table.parquet` and `training_config.json` (07); `oof_lgb_env.parquet` and `lgb_env_models/` (09).

**Method:**
1. Same pixel-level data (21,128 pixels) and the same 5 spatial folds as notebooks 08 to 10.
2. Train a second set of 16 LightGBM models on all 81 features (environment, Sentinel, MODIS) with the same parameters as notebook 09, and compare their out-of-fold predictions with the environment-only models (SPAM agreement).
3. SHAP (TreeExplainer) for all 16 crops, for the environment-only models and, if they are better, the all-feature models. Values are summed within feature families.
4. Check whether irrigation-related features (for example soil moisture, NDVI amplitude, salinity) matter for rice, onion and tomato.

**Outputs** (Google Drive): `oof_lgb_all.parquet`, `results_lgb_all.csv`, `lgb_all_models/`; SHAP tables in later cells.

**Notes:**
- Environment-only stays the main feature set unless the all-feature models are clearly better out of fold.
- SHAP explains the model, not the cause: correlated features (35 pairs with |r| > 0.95) share credit, so grouping by family is more reliable than single-feature rankings.
- No country-level SPAM irrigation feature is added here. It comes from the same source as the labels and could inflate SPAM agreement. It is tested only if SHAP shows irrigation-related features matter.

**Results:**

| Method | Mean rank correlation | Top-5 overlap | Top-1 match | Distinct top-5 sets |
|---|---|---|---|---|
| Observed SPAM | 1.000 | 1.000 | 1.000 | 430 |
| LightGBM, environment only (51) | 0.675 | 0.794 | 0.590 | 203 |
| LightGBM, all features (81) | 0.677 | 0.795 | 0.591 | 210 |

- Adding Sentinel and MODIS features changes rank correlation by +0.002 on average (12 of 16 crops up, 4 down; range -0.008 to +0.010). The difference is within noise, so the environment-only set is kept.
- SHAP (environment-only models, 4,000 pixels): climate carries the largest share of mean |SHAP| for every crop (37% to 77%), then soil (15% to 47%), terrain (4% to 14%) and salinity (under 4% for 15 of 16 crops). Solar radiation (`srad_mean`) is the top feature for 10 crops and in the top 5 for 15; elevation and topsoil bulk density also recur. These are likely geographic proxies and not agronomic causes, so the explanations name them as model drivers only.
- Irrigation: no irrigation feature is in the model. Dryness (`vpd_mean`) ranks high for onion, tomato and rice, but this reflects hot dry areas, not detected irrigation. Salinity ranks last. The SPAM country-level irrigation feature stays deferred.
- Limits: correlated features share credit (35 pairs with |r| > 0.95), so family-level reading is more reliable than single features. Sugarcane, yams and sweet potato models are weak, so their explanations are indicative only.
- Direction: the sign of `srad_mean` flips between crops (higher raises the score for sorghum, pearl millet, sesame, dry bean and sweet potato; lower raises it for cowpea, maize, rice, cassava, yams, onion, soybean, tomato and sugarcane), which shows it works as a north-south gradient and not as a crop requirement. Per feature, terrain (elevation) is as important as climate for sorghum, groundnut, rice, soybean and cotton; soil is more important per feature for cassava, beans and sugarcane. Explanations are shown as model associations only.

In [ ]:
# @title Cell 1: load, pixel table, feature families
!pip -q install shap lightgbm
import os, json, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
cfg = json.load(open(f'{P}/training_config.json'))
FEATURES, GROUPS, CROPS = cfg['features'], cfg['groups'], cfg['crops']
ENV = GROUPS['climate'] + GROUPS['soil'] + GROUPS['terrain'] + GROUPS['salinity']
SHARE = [f'share_{c}' for c in CROPS]
FAMILY = {f: fam for fam, fs in GROUPS.items() for f in fs}

tab = pd.read_parquet(f'{P}/training_table.parquet',
                      columns=['px', 'py', 'fold', 'adm0', 'has_label'] + FEATURES + SHARE)
lab = tab[tab.has_label]
g = lab.groupby(['px', 'py'])
pix = (g[FEATURES].mean().join(g[SHARE].first()).join(g[['fold']].first())
       .join(g['adm0'].agg(lambda s: s.mode().iat[0])).reset_index())
Y = np.clip(pix[SHARE].to_numpy('float64'), 0, 1)
folds = pix.fold.to_numpy()

print('Pixels:', len(pix), '| all features:', len(FEATURES), '| environment features:', len(ENV), '| crops:', len(CROPS))
print('Features per family:', {k: len(v) for k, v in GROUPS.items()})
print('Features without a family:', [f for f in FEATURES if f not in FAMILY])
print('Pixels per fold:', np.bincount(folds))

In [ ]:
# @title Cell 2: LightGBM on all 81 features (same folds and parameters), compared with environment-only
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

PARAMS = dict(objective='cross_entropy', learning_rate=0.05, n_estimators=300, num_leaves=31,
              min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
              reg_lambda=5.0, n_jobs=-1, random_state=42, verbose=-1)   # identical to notebook 09

K = 5
def top_mask(M, k=K):
    idx = np.argsort(-M, axis=1, kind='stable')[:, :k]
    m = np.zeros(M.shape, dtype=bool)
    np.put_along_axis(m, idx, True, axis=1)
    return m, idx

def summarize(Pr, name):
    ob, _ = top_mask(Y); pb, pi = top_mask(Pr)
    sets = pd.Series([tuple(sorted(r)) for r in pi]).value_counts()
    rho = np.nanmean([spearmanr(Y[:, j], Pr[:, j]).correlation for j in range(Y.shape[1])])
    return {'method': name, 'mean_spearman': round(float(rho), 3),
            'top5_overlap': round(float((ob & pb).sum(1).mean() / K), 3),
            'top1_match': round(float((np.argmax(Pr, 1) == np.argmax(Y, 1)).mean()), 3),
            'distinct_top5_sets': len(sets),
            'crops_in_top5_over_1pct': int((pb.mean(0) >= 0.01).sum())}

# Environment-only out-of-fold predictions from notebook 09
o = pd.read_parquet(f'{P}/oof_lgb_env.parquet')
P_env = pix[['px', 'py']].merge(o, on=['px', 'py'], how='left')[CROPS].to_numpy('float64')
print('Env-only predictions loaded:', P_env.shape, '| missing:', int(np.isnan(P_env).sum()))

# All-feature models, same 5 folds
X_all = pix[FEATURES]
P_all = np.zeros_like(Y)
for j, c in enumerate(CROPS):
    t0 = time.time()
    for k in range(5):
        tr, te = folds != k, folds == k
        m = lgb.LGBMRegressor(**PARAMS).fit(X_all[tr], Y[tr, j])
        P_all[te, j] = m.predict(X_all[te])
    print(f'{c}: {time.time() - t0:.0f} s')

out = pix[['px', 'py', 'fold']].copy(); out[CROPS] = P_all
out.to_parquet(f'{P}/oof_lgb_all.parquet', index=False)

# Comparison
res = pd.DataFrame([summarize(Y, 'observed SPAM (reference)'),
                    summarize(P_env, 'LightGBM: environment only (51)'),
                    summarize(P_all, 'LightGBM: all features (81)')]).set_index('method')
pd.set_option('display.width', 220)
print('\n', res.to_string())

def rho(Pr, j): return spearmanr(Y[:, j], Pr[:, j]).correlation
pc = pd.DataFrame({'mean_share': Y.mean(0),
                   'rho_env': [rho(P_env, j) for j in range(len(CROPS))],
                   'rho_all': [rho(P_all, j) for j in range(len(CROPS))],
                   'r2_env': [r2_score(Y[:, j], P_env[:, j]) for j in range(len(CROPS))],
                   'r2_all': [r2_score(Y[:, j], P_all[:, j]) for j in range(len(CROPS))]}, index=CROPS)
pc['d_rho'] = pc.rho_all - pc.rho_env
print('\nPer crop:'); print(pc.round(3))
print('\nCrops where all features improve rank correlation:', int((pc.d_rho > 0).sum()), 'of', len(CROPS))
print('Largest gains:', pc.d_rho.sort_values(ascending=False).head(3).round(3).to_dict())
print('Largest losses:', pc.d_rho.sort_values().head(3).round(3).to_dict())
res.to_csv(f'{P}/results_lgb_all.csv')
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 3: SHAP for the environment-only models, grouped by feature family
import shap

N_SHAP = 4000
rng = np.random.default_rng(0)
sel = rng.choice(len(pix), N_SHAP, replace=False)
X_env = pix[ENV]
Xs = X_env.iloc[sel]
FAM_ENV = ['climate', 'soil', 'terrain', 'salinity']
fam_of = np.array([FAMILY[f] for f in ENV])

models_env, shap_vals = {}, {}
t0 = time.time()
for j, c in enumerate(CROPS):
    m = lgb.LGBMRegressor(**PARAMS).fit(X_env, Y[:, j])
    models_env[c] = m
    sv = shap.TreeExplainer(m.booster_).shap_values(Xs)
    shap_vals[c] = np.asarray(sv)
    print(f'{c}: {time.time() - t0:.0f} s')

# Mean |SHAP| per feature and per family (share of total, %)
feat_imp = pd.DataFrame({c: np.abs(shap_vals[c]).mean(0) for c in CROPS}, index=ENV)
fam_imp = feat_imp.groupby(fam_of).sum().loc[FAM_ENV]
fam_share = (fam_imp / fam_imp.sum() * 100).round(1)
pd.set_option('display.width', 250)
print('\nShare of total mean |SHAP| by family (%), per crop:')
print(fam_share.T.to_string())

print('\nTop 5 features per crop (mean |SHAP|, model output scale):')
for c in CROPS:
    top = feat_imp[c].sort_values(ascending=False).head(5)
    print(f'{c}: ' + ', '.join(f'{f} ({v:.3f})' for f, v in top.items()))

# Irrigation / water-related check for rice, onion, tomato
WATER = [f for f in ENV if any(k in f for k in ['soil_moisture', 'era5_soilwater', 'sal_', 'def_annual', 'aet_annual', 'vpd'])]
print('\nWater-related features: rank (of 51) for rice, onion, tomato')
rk = feat_imp.rank(ascending=False).loc[WATER, ['rice', 'onio', 'toma']].astype(int)
print(rk.to_string())

feat_imp.to_csv(f'{P}/shap_feature_importance_env.csv')
fam_share.to_csv(f'{P}/shap_family_share_env.csv')
import joblib
joblib.dump({'sel': sel, 'shap': shap_vals}, '/content/shap_env_tmp.joblib', compress=3)
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 4: per-crop SHAP summary (top features, direction, per-feature family share)
import joblib
from scipy.stats import spearmanr
if 'shap_vals' not in globals():
    d = joblib.load('/content/shap_env_tmp.joblib')       # only exists if the session was not reset
    sel, shap_vals = d['sel'], d['shap']
    Xs = pix[ENV].iloc[sel]

rows = []
for c in CROPS:
    sv = shap_vals[c]; imp = np.abs(sv).mean(0)
    for rank, i in enumerate(np.argsort(-imp)[:10], 1):
        f = ENV[i]
        r = spearmanr(Xs[f], sv[:, i], nan_policy='omit').correlation
        direction = 'higher raises score' if r > 0.3 else 'lower raises score' if r < -0.3 else 'mixed / non-monotonic'
        rows.append({'crop': c, 'rank': rank, 'feature': f, 'family': FAMILY[f],
                     'mean_abs_shap': round(float(imp[i]), 4), 'direction': direction, 'rho': round(float(r), 2)})
top = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
print(top[top['rank'] <= 3].to_string(index=False))

# Per-feature family share: family share divided by the number of features in the family
nfeat = pd.Series({k: len(GROUPS[k]) for k in FAM_ENV})
per_feat = (fam_share.T / nfeat).round(2)                 # % of total |SHAP| per feature, crops x families
print('\nShare per feature (% of crop total |SHAP| per feature in the family):')
print(per_feat.to_string())

print('\nTop-1 feature counts:', top[top['rank'] == 1].feature.value_counts().to_dict())
print('Crops with srad_mean in top 5:', int(top[(top.feature == 'srad_mean') & (top['rank'] <= 5)].crop.nunique()))

top.to_csv(f'{P}/shap_top_features_env.csv', index=False)
per_feat.to_csv(f'{P}/shap_family_share_per_feature_env.csv')
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 5: heatmap of mean |SHAP| share by feature family (crops x families)
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, json, os
from google.colab import drive
drive.mount('/content/drive')

P = '/content/drive/MyDrive/crop_project'
os.makedirs(f'{P}/figures', exist_ok=True)
CROPS = json.load(open(f'{P}/training_config.json'))['crops']

fs = pd.read_csv(f'{P}/shap_family_share_env.csv', index_col=0)      # rows: families, columns: crops
H = fs.T.loc[CROPS]                                                    # crops x families
H = H.loc[H['climate'].sort_values(ascending=False).index]            # order crops by climate share

fig, ax = plt.subplots(figsize=(6.2, 6.5))
im = ax.imshow(H.values, cmap='Blues', vmin=0, vmax=max(60, H.values.max()), aspect='auto')
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns)
ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                color='white' if v > 35 else 'black', fontsize=9)
ax.set_title('Share of mean |SHAP| by feature family (%)\nLightGBM, environment-only, 4,000 pixels', fontsize=10)
cb = fig.colorbar(im, ax=ax, fraction=0.04); cb.set_label('% of total |SHAP| for the crop')
ax.set_xlabel('Feature family (sums over 23 climate, 18 soil, 5 terrain, 5 salinity features)', fontsize=8)
plt.tight_layout()
plt.savefig(f'{P}/figures/shap_family_heatmap.png', dpi=200)
plt.show()

In [ ]:
# @title Cell 6: env-only vs all-81 features: SPAM rank correlation and survey AUC with bootstrap intervals
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

CROPS = json.load(open(f'{P}/training_config.json'))['crops']
SHARE = [f'share_{c}' for c in CROPS]
tt = pd.read_parquet(f'{P}/training_table.parquet', columns=['px', 'py', 'has_label'] + SHARE)
pix_ = (tt[tt.has_label].groupby(['px', 'py'])[SHARE].first().reset_index())
Y_ = np.clip(pix_[SHARE].to_numpy('float64'), 0, 1)

def load_oof(name):
    o = pd.read_parquet(f'{P}/{name}.parquet').drop(columns='fold')
    return pix_[['px', 'py']].merge(o, on=['px', 'py'], how='left')[CROPS].to_numpy('float64')
MODELS = {'Environment only (51)': load_oof('oof_lgb_env'), 'All features (81)': load_oof('oof_lgb_all')}
COL = {'Environment only (51)': '#1f77b4', 'All features (81)': '#d95f02'}

# ---- Panel A: mean rank correlation with SPAM, bootstrap over pixels ----
# Shortcut: ranks computed once on the full sample, Pearson on resampled ranks (close to Spearman).
RY = np.column_stack([rankdata(Y_[:, j]) for j in range(Y_.shape[1])])
RP = {k: np.column_stack([rankdata(M[:, j]) for j in range(M.shape[1])]) for k, M in MODELS.items()}
def mean_rho(R1, R2, ix):
    a, b = R1[ix] - R1[ix].mean(0), R2[ix] - R2[ix].mean(0)
    return np.mean((a * b).sum(0) / np.sqrt((a ** 2).sum(0) * (b ** 2).sum(0)))
rng = np.random.default_rng(0); n = len(Y_); B = 300
boot = {k: [] for k in MODELS}
for _ in range(B):
    ix = rng.integers(0, n, n)
    for k in MODELS:
        boot[k].append(mean_rho(RY, RP[k], ix))
est = {k: mean_rho(RY, RP[k], np.arange(n)) for k in MODELS}
ci = {k: np.percentile(boot[k], [2.5, 97.5]) for k in MODELS}
diff = np.array(boot['All features (81)']) - np.array(boot['Environment only (51)'])
print('Mean rank correlation:', {k: round(v, 3) for k, v in est.items()})
print('Paired difference (all minus env): mean %.4f, 95%% interval [%.4f, %.4f]' % (diff.mean(), *np.percentile(diff, [2.5, 97.5])))

# ---- Panel B: survey presence AUC with bootstrap (same construction as 09b) ----
sites = pd.read_parquet(f'{P}/ground_truth_surveys_sites.parquet')
scr = pd.read_parquet(f'{P}/ground_truth_surveys_crops.parquet')
use = sites[sites.within_5km & (sites.has_label == True)].copy()
use['pixel'] = use.px.astype(int).astype(str) + '_' + use.py.astype(int).astype(str)
idx_ = {f'{int(a)}_{int(b)}': i for i, (a, b) in enumerate(zip(pix_.px, pix_.py))}
use = use[use.pixel.isin(idx_)]
full = (use[['site_id', 'country', 'pixel']].assign(_k=1)
        .merge(pd.DataFrame({'crop_code': CROPS, '_k': 1}), on='_k').drop(columns='_k')
        .merge(scr[['site_id', 'crop_code', 'weight_share']], on=['site_id', 'crop_code'], how='left')
        .fillna({'weight_share': 0.0}))
obs = full.groupby(['country', 'pixel', 'crop_code']).weight_share.mean().unstack('crop_code')[CROPS]

rows, rng = [], np.random.default_rng(1)
for country in ['Niger', 'Nigeria']:
    o = obs.xs(country, level='country'); ii = np.array([idx_[p] for p in o.index])
    for j, c in enumerate(CROPS):
        y = o[c].to_numpy() > 0
        if y.sum() < 10 or (~y).sum() < 10: continue
        bs = [rng.integers(0, len(y), len(y)) for _ in range(500)]          # same resamples for both models
        for k, M in MODELS.items():
            s = M[ii, j]
            v = [roc_auc_score(y[b], s[b]) for b in bs if 0 < y[b].sum() < len(y)]
            lo, hi = np.percentile(v, [2.5, 97.5])
            rows.append({'model': k, 'label': f'{country[:3]} {c}', 'auc': roc_auc_score(y, s), 'lo': lo, 'hi': hi})
S = pd.DataFrame(rows)
order = S[S.model == 'Environment only (51)'].sort_values('auc').label.tolist()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={'width_ratios': [1, 2.2]})
for i, k in enumerate(MODELS):
    a1.errorbar(i, est[k], yerr=[[est[k] - ci[k][0]], [ci[k][1] - est[k]]], fmt='o', color=COL[k], capsize=5, ms=8)
a1.set_xticks(range(2)); a1.set_xticklabels(['Env only\n(51)', 'All\n(81)'])
a1.set_xlim(-0.6, 1.6); a1.set_ylabel('Mean rank correlation with SPAM (16 crops)')
a1.set_title('A. Agreement with SPAM\n(95% bootstrap over pixels)', fontsize=10)
a1.grid(axis='y', alpha=.3)
for k, off in zip(MODELS, [-0.13, 0.13]):
    d = S[S.model == k].set_index('label').loc[order]
    y = np.arange(len(d)) + off
    a2.errorbar(d.auc, y, xerr=[d.auc - d.lo, d.hi - d.auc], fmt='o', color=COL[k], capsize=2, ms=5, label=k)
a2.axvline(0.5, color='grey', ls='--', lw=1)
a2.set_yticks(range(len(order))); a2.set_yticklabels(order)
a2.set_xlabel('Presence AUC'); a2.set_title('B. Survey presence AUC (Niger, Nigeria; 95% bootstrap over pixels)', fontsize=10)
a2.legend(loc='lower right', fontsize=8); a2.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.savefig(f'{P}/figures/env_vs_all_comparison.png', dpi=200); plt.show()